# JAX Examples

## Basic JAX Benchmarking

ZeroPyBench automatically detects JAX arrays and optimizes benchmarking:

:::{note}
ZeroPyBench automatically detects JAX arrays in your code and wraps the benchmarked expression in a JIT-compiled function. The compilation time is measured separately.
:::

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr

from zeropybench import Benchmark

bench = Benchmark(repeat=20)
x = jnp.ones(1000)
y = jnp.ones(1000)

with bench():
    x + y

## Verbose Mode

Use `verbose=True` to see the setup code (JIT-compiled function) and the benchmarked code:

In [ ]:
bench = Benchmark(verbose=True)

with bench():
    x + y

For functions returning a PyTree (e.g., a tuple of Arrays), the slightly slower `jax.block_until_ready` is used instead.

In [ ]:
from dataclasses import dataclass


@jax.tree_util.register_dataclass
@dataclass
class Vector:
    x: jax.Array
    y: jax.Array

    @classmethod
    def normal(cls, key, shape):
        key_x, key_y = jr.split(key)
        return Vector(jr.normal(key_x, shape), jr.normal(key_y, shape))

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)


key1, key2 = jr.split(jr.key(42))
shape = (10_000,)
v1 = Vector.normal(key1, shape)
v2 = Vector.normal(key2, shape)

bench = Benchmark(verbose=True)
with bench():
    v1 + v2

## JAX-Specific Report Fields

When JAX code is detected, the benchmark report includes additional fields:

In [ ]:
report = bench.to_dicts()[0]
print('Available fields:', list(report.keys()))

In [ ]:
print(f'First execution time: {1e6 * report["first_execution_time"]:.2f} µs')
print(f'Compilation time: {1e6 * report["compilation_time"]:.2f} µs')
print(f'Median execution time: {1e6 * report["median_execution_time"]:.2f} µs')

## Visualizing the HLO

The `hlo` field contains the StableHLO representation of the compiled function. You can visualize it using `visu_hlo`:

In [ ]:
from visu_hlo import show

hlo_text = report['hlo']
show(hlo_text)

## Comparing Broadcasting Strategies

Benchmark different array operations to compare their performance:

In [ ]:
bench = Benchmark()

for N in [100, 1000, 10000]:
    x = jnp.ones(N)
    y = jnp.ones(1000)

    with bench(method='broadcast right', N=N):
        x[:, None] + y[None, :]

    with bench(method='broadcast left', N=N):
        x[None, :] + y[:, None]

In [ ]:
print(bench)

## Benchmarking JIT-compiled Functions

ZeroPyBench handles both regular and JIT-compiled JAX functions:

:::{note}
When benchmarking an already JIT-compiled function, ZeroPyBench reuses it directly without re-wrapping, preserving any `static_argnums` or other JIT options you specified.
:::

In [ ]:
import jax


@jax.jit
def matmul(a, b):
    return a @ b


bench = Benchmark()

for N in [64, 128, 256, 512]:
    a = jnp.ones((N, N))
    b = jnp.ones((N, N))

    with bench(operation='matmul', N=N):
        matmul(a, b)

In [ ]:
print(bench)

## Visualizing the Matrix Multiplication HLO

In [ ]:
# Get the HLO for the first benchmark (N=64)
report = bench.to_dicts()[0]
show(report['hlo'])

## Plotting Results

In [ ]:
bench.plot()